# Autonomy, Self-Direction, and Termination

How much should an agent decide on its own? **Autonomy** is the agent's freedom to choose tools, retry strategies, and when to stop. **Self-direction** is the model routing its own next step inside guardrails. **Termination** is how and why the loop ends — successfully, by budget, or by guardrail.

Too little autonomy feels rigid. Too much produces runaway loops, surprise costs, and actions nobody approved. Production systems set **autonomy tiers**, **hard termination rules**, and **progress detectors** the model cannot override.

This notebook implements a **ShopCo Support Investigator** — a self-directed agent that looks up orders, searches policy, and opens tickets — with configurable autonomy levels and a full termination engine. Everything is self-contained plain Python.

In [ ]:
"""
ShopCo Support Investigator — autonomy and termination demo.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
    {"id": "esc-05", "text": "Escalate to human if order not found after 2 lookup attempts."},
]

TICKETS: list[dict[str, Any]] = []

print(f"Backend ready — {len(ORDERS)} orders, {len(POLICIES)} policies")

---

## 1. Autonomy Spectrum — How Much Freedom?

| Level | Name | Who decides next step | ShopCo example |
|-------|------|----------------------|----------------|
| **L0** | Scripted | Code only | Fixed: lookup → policy → answer |
| **L1** | Tool pick | Model picks one tool from allowlist | Choose lookup vs policy search |
| **L2** | Multi-step | Model loops until `finish` | Investigate, retry, then answer |
| **L3** | Write autonomy | Model can create tickets (with caps) | Open ticket after failed lookup |
| **L4** | Full autonomy | Model plans freely (rare in prod) | Unbounded — **avoid** without HITL |

Most production agents operate at **L1–L2** with orchestrator-enforced termination. **L3** requires approval gates for side effects.

---

## 2. Self-Direction vs Orchestrator Control

```
┌─────────────────────────────────────────────────────────────┐
│  Orchestrator (your code — always in control)               │
│  ├── AutonomyConfig (max steps, allowed tools, budgets)     │
│  ├── TerminationEngine (checks EVERY iteration)             │
│  └── ProgressDetector (stall / repeat detection)            │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│  Self-directed agent loop                                   │
│  Model proposes: tool_call | finish | ask_clarification       │
│  Runtime disposes: validate → execute → observe → repeat    │
└─────────────────────────────────────────────────────────────┘
```

**Self-direction** means the model proposes the next action. **Orchestrator control** means your code validates, executes, and can **force termination** regardless of what the model wants.

---

## 3. Termination — Why the Loop Stops

| Signal | Who triggers | Example |
|--------|--------------|--------|
| **final_answer** | Agent calls `finish` | "Order ORD-1001 is shipped" |
| **max_iterations** | Orchestrator | step_count >= 6 |
| **max_tool_calls** | Orchestrator | 20 tools invoked |
| **cost_cap** | Orchestrator | estimated_cost >= $0.50 |
| **guardrail_block** | Policy engine | Injection or blocked action |
| **no_progress** | Progress detector | Same tool+args 3 times |
| **error_budget** | Orchestrator | 3 consecutive tool errors |
| **human_cancel** | HITL gate | User rejects ticket creation |
| **clarification_needed** | Agent | Missing order_id — ask user |

Every run should end with a **TerminationRecord** — reason, detail, and partial state if stopped early.

In [ ]:
class TerminationReason(str, Enum):
    FINAL_ANSWER = "final_answer"
    MAX_ITERATIONS = "max_iterations"
    MAX_TOOL_CALLS = "max_tool_calls"
    COST_CAP = "cost_cap"
    GUARDRAIL_BLOCK = "guardrail_block"
    NO_PROGRESS = "no_progress"
    ERROR_BUDGET = "error_budget"
    HUMAN_CANCEL = "human_cancel"
    CLARIFICATION_NEEDED = "clarification_needed"


@dataclass
class TerminationRecord:
    reason: TerminationReason
    detail: str = ""
    step: int = 0
    tool_calls: int = 0
    cost_usd: float = 0.0


@dataclass
class AutonomyConfig:
    level: int = 2
    max_iterations: int = 6
    max_tool_calls: int = 12
    max_cost_usd: float = 0.25
    cost_per_step_usd: float = 0.003
    max_consecutive_errors: int = 3
    max_repeat_actions: int = 2
    allowed_tools: list[str] = field(default_factory=lambda: [
        "lookup_order", "search_policy", "create_ticket", "finish", "ask_clarification"
    ])
    allow_writes: bool = True


print("Termination reasons:", [r.value for r in TerminationReason])
print("Default config:", AutonomyConfig())

---

## 4. Tools — Investigator Actions

Self-directed agents need a **`finish`** tool (or equivalent) so the model can signal completion. Optional **`ask_clarification`** prevents guessing when context is missing.

In [ ]:
def lookup_order(order_id: str) -> str:
    oid = order_id.upper().strip()
    if not re.match(r"^ORD-\d{4}$", oid):
        return f"[STATUS: error] Invalid order_id '{order_id}'"
    order = ORDERS.get(oid)
    if not order:
        return f"[STATUS: error] Order {oid} not found"
    return f"[STATUS: ok] {json.dumps({'order_id': oid, **order})}"


def search_policy(query: str) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = [p for p in POLICIES if any(t in p["text"].lower() for t in terms)]
    return f"[STATUS: ok] {json.dumps(hits or POLICIES[:1])}"


def create_ticket(customer_email: str, subject: str, body: str) -> str:
    if "@" not in customer_email:
        return "[STATUS: error] Invalid email"
    ticket = {
        "ticket_id": f"TKT-{uuid.uuid4().hex[:6].upper()}",
        "customer": customer_email,
        "subject": subject[:100],
        "body": body[:300],
    }
    TICKETS.append(ticket)
    return f"[STATUS: ok] {json.dumps(ticket)}"


TOOL_IMPL: dict[str, Callable[..., str]] = {
    "lookup_order": lookup_order,
    "search_policy": search_policy,
    "create_ticket": create_ticket,
}

print(lookup_order("ORD-1001"))

---

## 5. Termination Engine — Checks Every Step

The orchestrator runs termination checks **before and after** each model turn. The model cannot override `max_iterations` or `cost_cap`.

In [ ]:
@dataclass
class RunBudget:
    step: int = 0
    tool_calls: int = 0
    cost_usd: float = 0.0
    consecutive_errors: int = 0
    action_fingerprints: list[str] = field(default_factory=list)

    def record_step(self, cfg: AutonomyConfig) -> None:
        self.step += 1
        self.cost_usd += cfg.cost_per_step_usd

    def record_tool(self, name: str, args: dict, success: bool) -> None:
        self.tool_calls += 1
        fp = f"{name}:{json.dumps(args, sort_keys=True)}"
        self.action_fingerprints.append(fp)
        if success:
            self.consecutive_errors = 0
        else:
            self.consecutive_errors += 1


class TerminationEngine:
    def check(self, budget: RunBudget, cfg: AutonomyConfig) -> TerminationRecord | None:
        if budget.step >= cfg.max_iterations:
            return TerminationRecord(TerminationReason.MAX_ITERATIONS, f"step={budget.step}", budget.step, budget.tool_calls, budget.cost_usd)
        if budget.tool_calls >= cfg.max_tool_calls:
            return TerminationRecord(TerminationReason.MAX_TOOL_CALLS, f"calls={budget.tool_calls}", budget.step, budget.tool_calls, budget.cost_usd)
        if budget.cost_usd >= cfg.max_cost_usd:
            return TerminationRecord(TerminationReason.COST_CAP, f"${budget.cost_usd:.4f}", budget.step, budget.tool_calls, budget.cost_usd)
        if budget.consecutive_errors >= cfg.max_consecutive_errors:
            return TerminationRecord(TerminationReason.ERROR_BUDGET, f"errors={budget.consecutive_errors}", budget.step, budget.tool_calls, budget.cost_usd)
        if self._detect_no_progress(budget, cfg):
            return TerminationRecord(TerminationReason.NO_PROGRESS, "repeated identical actions", budget.step, budget.tool_calls, budget.cost_usd)
        return None

    def _detect_no_progress(self, budget: RunBudget, cfg: AutonomyConfig) -> bool:
        if len(budget.action_fingerprints) < cfg.max_repeat_actions + 1:
            return False
        last = budget.action_fingerprints[-1]
        repeats = sum(1 for fp in budget.action_fingerprints if fp == last)
        return repeats > cfg.max_repeat_actions


engine = TerminationEngine()
demo_budget = RunBudget(step=6, tool_calls=2, cost_usd=0.02)
result = engine.check(demo_budget, AutonomyConfig(max_iterations=6))
print(f"At max steps: {result.reason.value if result else 'continue'}")

---

## 6. Self-Directed Model — Offline Stand-In

`SelfDirectedModel` simulates an L2 agent that picks tools based on conversation state, then calls `finish` when enough evidence is gathered.

In [ ]:
class SelfDirectedModel:
    """Rule-based stand-in for a self-routing LLM."""

    def propose(self, messages: list[dict[str, Any]], cfg: AutonomyConfig) -> dict[str, Any]:
        tool_msgs = [m for m in messages if m.get("role") == "tool"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")

        if cfg.level == 0:
            return self._scripted_l0(tool_msgs, user_msg)
        return self._autonomous_l2(tool_msgs, user_msg, cfg)

    def _scripted_l0(self, tool_msgs: list, user_msg: str) -> dict[str, Any]:
        if len(tool_msgs) == 0:
            oid = re.search(r"ORD-\d{4}", user_msg.upper())
            return {"action": "tool", "name": "lookup_order", "args": {"order_id": oid.group(0) if oid else "ORD-1001"}}
        if len(tool_msgs) == 1:
            return {"action": "tool", "name": "search_policy", "args": {"query": "returns"}}
        return {"action": "finish", "answer": self._summarize(tool_msgs)}

    def _autonomous_l2(self, tool_msgs: list, user_msg: str, cfg: AutonomyConfig) -> dict[str, Any]:
        if not tool_msgs:
            if "ord-" not in user_msg.lower() and "order" not in user_msg.lower():
                if "return" in user_msg.lower() or "policy" in user_msg.lower():
                    return {"action": "tool", "name": "search_policy", "args": {"query": user_msg[:60]}}
                return {"action": "clarify", "question": "Please provide your order ID (e.g. ORD-1001)."}
            oid = re.search(r"ORD-\d{4}", user_msg.upper())
            return {"action": "tool", "name": "lookup_order", "args": {"order_id": oid.group(0) if oid else "ORD-1001"}}

        last_obs = tool_msgs[-1]["content"]
        if "[STATUS: error]" in last_obs and "not found" in last_obs.lower():
            if cfg.allow_writes and cfg.level >= 3:
                return {
                    "action": "tool", "name": "create_ticket",
                    "args": {
                        "customer_email": "customer@shop.com",
                        "subject": "Order not found — investigation",
                        "body": user_msg[:200],
                    },
                }
            return {"action": "finish", "answer": "I could not find that order. Please verify the order ID or contact support."}

        if len(tool_msgs) == 1 and any(w in user_msg.lower() for w in ("return", "cancel")):
            return {"action": "tool", "name": "search_policy", "args": {"query": "return cancel"}}

        return {"action": "finish", "answer": self._summarize(tool_msgs)}

    def _summarize(self, tool_msgs: list) -> str:
        parts = [m["content"].replace("[STATUS: ok] ", "")[:80] for m in tool_msgs]
        return "Based on my investigation: " + " | ".join(parts)


model = SelfDirectedModel()
print("Model ready")

---

## 7. Autonomous Investigator Loop

The core loop: propose → validate → terminate check → execute → observe → repeat.

In [ ]:
@dataclass
class InvestigatorResult:
    answer: str
    termination: TerminationRecord
    messages: list[dict[str, Any]]
    trace: list[str]


class ShopCoInvestigator:
    def __init__(self, cfg: AutonomyConfig | None = None):
        self.cfg = cfg or AutonomyConfig()
        self.model = SelfDirectedModel()
        self.engine = TerminationEngine()

    def run(self, user_query: str, thread_id: str = "inv-001") -> InvestigatorResult:
        TICKETS.clear()
        messages: list[dict[str, Any]] = [
            {"role": "system", "content": "ShopCo support investigator. Use tools to research, then finish."},
            {"role": "user", "content": user_query},
        ]
        budget = RunBudget()
        trace = [f"start:{thread_id} level=L{self.cfg.level}"]

        while True:
            budget.record_step(self.cfg)
            term = self.engine.check(budget, self.cfg)
            if term:
                trace.append(f"FORCED STOP: {term.reason.value}")
                return InvestigatorResult(
                    f"Investigation stopped ({term.reason.value}): {term.detail}",
                    term, messages, trace,
                )

            proposal = self.model.propose(messages, self.cfg)
            action = proposal.get("action")
            trace.append(f"step {budget.step}: propose={action}")

            if action == "finish":
                term = TerminationRecord(
                    TerminationReason.FINAL_ANSWER, "agent finish",
                    budget.step, budget.tool_calls, budget.cost_usd,
                )
                trace.append("terminate: final_answer")
                return InvestigatorResult(proposal["answer"], term, messages, trace)

            if action == "clarify":
                term = TerminationRecord(
                    TerminationReason.CLARIFICATION_NEEDED, proposal["question"],
                    budget.step, budget.tool_calls, budget.cost_usd,
                )
                trace.append("terminate: clarification_needed")
                return InvestigatorResult(proposal["question"], term, messages, trace)

            if action == "tool":
                name, args = proposal["name"], proposal["args"]
                if name not in self.cfg.allowed_tools:
                    term = TerminationRecord(TerminationReason.GUARDRAIL_BLOCK, f"tool {name} not allowed")
                    return InvestigatorResult(f"Blocked: {name} not in allowlist", term, messages, trace)
                if name == "create_ticket" and not self.cfg.allow_writes:
                    term = TerminationRecord(TerminationReason.GUARDRAIL_BLOCK, "writes disabled")
                    return InvestigatorResult("Ticket creation not permitted at this autonomy level", term, messages, trace)

                fn = TOOL_IMPL.get(name)
                obs = fn(**args) if fn else f"[STATUS: error] Unknown tool {name}"
                success = "[STATUS: ok]" in obs
                budget.record_tool(name, args, success)
                messages.append({"role": "assistant", "content": None, "tool_call": {"name": name, "args": args}})
                messages.append({"role": "tool", "content": obs})
                trace.append(f"  tool:{name} success={success}")
                continue

            term = TerminationRecord(TerminationReason.GUARDRAIL_BLOCK, f"unknown action {action}")
            return InvestigatorResult("Invalid agent action", term, messages, trace)


investigator = ShopCoInvestigator(AutonomyConfig(level=2))
r = investigator.run("Where is order ORD-1001 and can I return it?")
print(f"Answer: {r.answer[:100]}...")
print(f"Termination: {r.termination.reason.value} | steps={r.termination.step} | tools={r.termination.tool_calls}")

---

## 8. Compare Autonomy Levels L0 vs L2 vs L3

Same query, different autonomy configuration — observe steps, tools, and termination.

In [ ]:
QUERY = "Order ORD-9999 not found — please help"

CONFIGS = [
    AutonomyConfig(level=0, max_iterations=5, allow_writes=False),
    AutonomyConfig(level=2, max_iterations=6, allow_writes=False),
    AutonomyConfig(level=3, max_iterations=6, allow_writes=True),
]

print(f"{'Level':<6} {'Termination':<22} {'Steps':<6} {'Tools':<6} Answer preview")
print("-" * 85)
for cfg in CONFIGS:
    inv = ShopCoInvestigator(cfg)
    r = inv.run(QUERY)
    print(f"L{cfg.level:<5} {r.termination.reason.value:<22} {r.termination.step:<6} {r.termination.tool_calls:<6} {r.answer[:45]}...")

---

## 9. Forced Termination Demos

Orchestrator stops the run even if the model would continue.

In [ ]:
class StubbornModel:
    """Always proposes another lookup — never finishes."""

    def propose(self, messages: list, cfg: AutonomyConfig) -> dict:
        return {"action": "tool", "name": "lookup_order", "args": {"order_id": "ORD-1001"}}


class LoopInvestigator(ShopCoInvestigator):
    def __init__(self, cfg: AutonomyConfig):
        super().__init__(cfg)
        self.model = StubbornModel()


tight_cfg = AutonomyConfig(max_iterations=4, max_repeat_actions=2)
loop_inv = LoopInvestigator(tight_cfg)
r = loop_inv.run("stuck loop test")

print(f"Termination: {r.termination.reason.value}")
print(f"Detail: {r.termination.detail}")
print("Trace:")
for line in r.trace:
    print(f"  {line}")

---

## 10. Clarification as Termination

Self-directed agents should **ask** rather than guess when required context is missing. `CLARIFICATION_NEEDED` is a valid, successful termination — not a failure.

In [ ]:
r = ShopCoInvestigator(AutonomyConfig(level=2)).run("I want to return my purchase")
print(f"Reason: {r.termination.reason.value}")
print(f"Message: {r.answer}")
print(f"Steps used: {r.termination.step} (stopped early — good)")

---

## 11. Human-in-the-Loop Termination

At L3, write actions can require human approval before the loop continues.

In [ ]:
@dataclass
class HITLGate:
    auto_approve_reads: bool = True
    pending_writes: list[dict[str, Any]] = field(default_factory=list)

    def request_write(self, action: str, args: dict) -> str:
        proposal = {"action": action, "args": args, "id": len(self.pending_writes) + 1}
        self.pending_writes.append(proposal)
        return f"pending_approval:{proposal['id']}"

    def approve(self, proposal_id: int, approved: bool) -> bool:
        if proposal_id < 1 or proposal_id > len(self.pending_writes):
            return False
        return approved


class HITLInvestigator(ShopCoInvestigator):
    def __init__(self, cfg: AutonomyConfig, hitl: HITLGate):
        super().__init__(cfg)
        self.hitl = hitl

    def run(self, user_query: str, human_approves_write: bool = False) -> InvestigatorResult:
        result = super().run(user_query)
        if result.termination.reason == TerminationReason.GUARDRAIL_BLOCK and "create_ticket" in result.answer:
            pass
        return result


# Demo: L3 with write blocked until human approves
gate = HITLGate()
cfg_l3 = AutonomyConfig(level=3, allow_writes=False)  # writes need HITL wrapper
r = ShopCoInvestigator(cfg_l3).run("Order ORD-9999 missing")
print(f"Without write permission: {r.termination.reason.value}")
print(f"Answer: {r.answer[:80]}")

cfg_l3_allow = AutonomyConfig(level=3, allow_writes=True)
r2 = ShopCoInvestigator(cfg_l3_allow).run("Order ORD-9999 missing")
print(f"\nWith write permission: {r2.termination.reason.value} | tickets={len(TICKETS)}")

---

## 12. Termination Decision Flow

```
Each iteration:
    │
    ├─ budget.step >= max_iterations? ──► MAX_ITERATIONS
    ├─ budget.tool_calls >= max? ───────► MAX_TOOL_CALLS
    ├─ budget.cost >= cap? ─────────────► COST_CAP
    ├─ consecutive_errors >= budget? ───► ERROR_BUDGET
    ├─ same action repeated? ───────────► NO_PROGRESS
    │
    ▼
    Model proposes action
    │
    ├─ finish ──────────────────────────► FINAL_ANSWER
    ├─ clarify ─────────────────────────► CLARIFICATION_NEEDED
    ├─ tool not in allowlist ───────────► GUARDRAIL_BLOCK
    ├─ write blocked ───────────────────► GUARDRAIL_BLOCK / escalate
    └─ execute tool ────────────────────► loop
```

---

## 13. Batch Runs — Termination Statistics

Track how often each termination reason fires — essential for tuning autonomy config.

In [ ]:
TEST_QUERIES = [
    "Where is ORD-1001?",
    "Return policy?",
    "I want to return something",
    "ORD-9999 help",
    "ORD-1002 cancel before ship?",
]

stats: dict[str, int] = {}
inv = ShopCoInvestigator(AutonomyConfig(level=2))

print(f"{'Query':<35} {'Reason':<22} Steps Tools")
print("-" * 70)
for q in TEST_QUERIES:
    r = inv.run(q)
    reason = r.termination.reason.value
    stats[reason] = stats.get(reason, 0) + 1
    print(f"{q[:35]:<35} {reason:<22} {r.termination.step:<5} {r.termination.tool_calls}")

print("\nTermination distribution:", stats)

---

## 14. Tuning Autonomy for Production

| Knob | Too low | Too high |
|------|---------|----------|
| `max_iterations` | Gives up on complex cases | Runaway loops |
| `max_cost_usd` | Frequent cost_cap stops | Bill shock |
| `max_repeat_actions` | Stops before useful retry | Wastes steps on stuck agent |
| `allowed_tools` | Cannot complete task | Over-privileged actions |
| `allow_writes` | Cannot escalate | Unapproved side effects |

**Start conservative** (low steps, read-only), measure termination distribution in evals, then loosen autonomy where agents consistently succeed.

---

## 15. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| No `finish` tool | Model never signals done | Add explicit termination action |
| Only max_tokens | Loop runs 100 tool calls in 5 steps | Cap tool_calls separately |
| Trust model to stop | Infinite loops | Orchestrator forced termination |
| Guessing missing context | Wrong order lookups | `ask_clarification` termination |
| L4 in production | Unbounded agency | L1–L2 + HITL for writes |
| No termination logging | Cannot tune budgets | `TerminationRecord` every run |

---

## 16. Optional — Live OpenAI Self-Directed Loop

Set `USE_LIVE_LLM = True` to replace `SelfDirectedModel` with real tool-calling.

In [ ]:
TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "lookup_order", "description": "Look up order by ORD-####", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}},
    {"type": "function", "function": {"name": "search_policy", "description": "Search policies", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "create_ticket", "description": "Create support ticket", "parameters": {"type": "object", "properties": {"customer_email": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}}, "required": ["customer_email", "subject", "body"]}}},
]


def make_live_proposer(cfg: AutonomyConfig):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def propose(messages: list[dict], _cfg: AutonomyConfig) -> dict:
        api_msgs = [{"role": m["role"], "content": m.get("content") or ""} for m in messages if m["role"] != "tool"]
        for m in messages:
            if m["role"] == "tool":
                api_msgs.append({"role": "tool", "content": m["content"], "tool_call_id": "tc"})
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=api_msgs, tools=TOOL_SCHEMAS, temperature=0)
        msg = resp.choices[0].message
        if msg.tool_calls:
            tc = msg.tool_calls[0]
            return {"action": "tool", "name": tc.function.name, "args": json.loads(tc.function.arguments)}
        return {"action": "finish", "answer": msg.content or "Done."}
    return propose


if USE_LIVE_LLM:
    live_inv = ShopCoInvestigator(AutonomyConfig(level=2, max_iterations=4))
    live_inv.model.propose = make_live_proposer(live_inv.cfg)  # type: ignore
    print(live_inv.run("Where is ORD-1001?").answer)
else:
    print("Offline mode — SelfDirectedModel used for all demos.")

---

## 17. Check Your Understanding

1. What is the difference between **self-direction** and **orchestrator control**?
2. Name four **termination reasons** besides `final_answer`.
3. Why is `CLARIFICATION_NEEDED` a valid termination, not a failure?
4. What does **no_progress** detection prevent?
5. When should you use L0 vs L2 autonomy?
6. Why cap **tool_calls** separately from **iterations**?
7. What must every run log in `TerminationRecord`?

---

## 18. Summary

Autonomy is a **dial**, not a default to max:

- **L0–L2** covers most production: scripted, tool-pick, or multi-step investigate-then-finish.
- **Self-direction** lets the model propose; the **orchestrator** validates and can **force stop**.
- **Termination** must be explicit: `finish`, budgets, guardrails, clarification, or human gate.
- **`TerminationEngine`** checks every step; **`RunBudget`** tracks steps, tools, cost, errors, repeats.
- Tune autonomy from **termination statistics** — not gut feel.

The ShopCo Investigator demonstrated successful completion, clarification stops, write escalation at L3, and forced stops on runaway loops. That is the control loop every autonomous agent needs before production.